# Determine (Ab)Normality of Current Values based on Past Values

Determine whether the current laboratory parameter value is normal or abnormal, based solely on the last measurement of the same parameter between 2 and 36 hours ago.

In [ ]:
import lab_ecg
import pandas as pd
import tqdm

In [ ]:
samples = pd.read_hdf("data/mimic4_samples.h5", key="table")

In [ ]:
dct = {}
for c in tqdm.tqdm(lab_ecg.MIMIC_LAB_MAPPING):
    sub = samples.loc[samples[c].notna(), [c, c + "_split", "subject_id", "sample_id", "ecg_time"]]
    sub = sub.join(
        lab_ecg.resample_eav(sub, "subject_id", c, "ecg_time", pd.Timedelta("-36 hours"), pd.Timedelta("-2 hours"))
    )
    sub = sub[sub[c + "_r-1"].notna() & (sub[c + "_split"] == "test")]

    for param, th, pretty in lab_ecg.TARGETS:
        if param == c:
            if th.startswith("<"):
                factor = -1
                y_true = sub[c] < float(th[1:])
                y_pred = sub[c + "_r-1"] < float(th[1:])  # use same threshold for predictions
            else:
                factor = 1
                y_true = sub[c] > float(th[1:])
                y_pred = sub[c + "_r-1"] > float(th[1:])  # use same threshold for predictions
            if y_true.nunique() == 2:
                avg = lab_ecg.evaluate(y_true, factor * sub[c + "_r-1"], y_pred, n_repetitions=1).agg("nanmean")
                aux = {k: f"{v:.3f}" for k, v in avg.items()}
            else:
                aux = {}
            aux["n_br"] = f"{len(y_true):d} ({y_true.mean() * 100:.2f}%)"
            dct[pretty] = aux

In [ ]:
pd.DataFrame.from_dict(dct, orient="index").sort_index()